# National AI Strategy — Text Analysis (DE, FR, IT, US)

Comparative text analysis of 4 National AI Strategy documents:

- `DE_National_AI_Strategy.pdf` — Germany
- `FR__National_AI_Strategy.pdf` — France
- `IT_National_AI_Strategy.pdf` — Italy
- `US_National_AI_Strategy.pdf` — United States

Located in [`data/pdf/AI Policy/`](../../data/pdf/AI%20Policy/).

## Required Packages

```bash
pip install pdfplumber scikit-learn matplotlib pandas wordcloud nltk
```


## 1. Setup

In [ ]:
from pathlib import Path
import re
from collections import Counter

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import pdfplumber
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import LatentDirichletAllocation

plt.rcParams['axes.unicode_minus'] = False


def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'data').is_dir():
            return p
    raise FileNotFoundError('Could not locate repo root containing `data/`')


repo_root = find_repo_root(Path.cwd())
pdf_dir = repo_root / 'data' / 'pdf' / 'AI Policy'

print(f'Repo root: {repo_root}')
print(f'PDF dir:   {pdf_dir}')
print(f'PDFs found: {list(pdf_dir.glob("*.pdf"))}')


## 2. Country Mapping

In [ ]:
COUNTRY_MAP = {
    'DE_National_AI_Strategy.pdf':  'Germany',
    'FR__National_AI_Strategy.pdf': 'France',
    'IT_National_AI_Strategy.pdf':  'Italy',
    'US_National_AI_Strategy.pdf':  'United States',
}

COLORS = {
    'Germany':       '#8c564b',
    'France':        '#9467bd',
    'Italy':         '#e377c2',
    'United States': '#1f77b4',
}


## 3. Extract Text from PDFs

Uses `pdfplumber`. Returns a dict `{country: full_text}`.


In [ ]:
def extract_pdf_text(path: Path) -> str:
    pages = []
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            t = page.extract_text() or ''
            pages.append(t)
    return '\n'.join(pages)


texts = {}
page_counts = {}

for pdf_path in sorted(pdf_dir.glob('*.pdf')):
    country = COUNTRY_MAP.get(pdf_path.name, pdf_path.stem)
    text = extract_pdf_text(pdf_path)
    texts[country] = text
    with pdfplumber.open(pdf_path) as p:
        page_counts[country] = len(p.pages)
    print(f'{country:15s}  {page_counts[country]:>4d} pages, {len(text):>8,} chars')


## 4. Document Statistics

In [ ]:
def word_tokenize(text: str) -> list:
    """Lowercase tokens of letters only (length >= 3)."""
    return re.findall(r"[a-zA-Zà-ÿÀ-ÝäöüÄÖÜß']{3,}", text.lower())


stats = []
for country, text in texts.items():
    tokens = word_tokenize(text)
    stats.append({
        'Country': country,
        'Pages': page_counts[country],
        'Characters': len(text),
        'Words (>=3 chars)': len(tokens),
        'Unique words': len(set(tokens)),
        'Type-token ratio': round(len(set(tokens)) / max(len(tokens), 1), 3),
    })

stats_df = pd.DataFrame(stats).set_index('Country')
print('Document statistics:')
stats_df


## 5. Build a Stopword List (multilingual)

In [ ]:
# Combined stopword list for EN, DE, FR, IT
EN_STOP = set('the a an and or but if is are was were be been being have has had do does did the of in to for on with at from by as it its this that these those we our us i you he she they them their there here can will would should may might shall must not no yes also more most some any all each every other another such own same so than then thus too very into over under between among through during before after above below upon out off down up'.split())
DE_STOP = set('der die das den dem des ein eine einen einer eines einem auch aber als alle alles am an auf aus bei bin bis dann dass dem den der des deshalb dich die dies dir doch dort durch ein eine einen einer eines er es etwas für gegen gewesen habe haben hatten hier ich ihm ihn ihnen ihre im in ins ist ja jede jeden jeder jedes jenen jener jenes kann kein keine keinen keiner keines könne konnten machen man mehr mit nach nicht noch nur ob oder ohne sehr sein seine sich sie sind so soll soweit sowie über um und uns unser unsere von vor war waren was weg wegen weil welche welchen welcher welches wenn wer werde werden wie wieder wir wird wirst wo woher wohin zu zum zur zwar zwischen'.split())
FR_STOP = set('le la les un une des de du au aux dans sur par pour avec sans sous chez ce cette ces cet est sont ai as a avons avez ont être étant été être avoir étant et ou mais donc car ne pas plus moins très tout tous toute toutes même mêmes je tu il elle ils elles nous vous mon ma mes ton ta tes son sa ses notre nos votre vos leur leurs qui que quoi dont où quand comment combien si lors lorsqu pourquoi quel quelle quels quelles ainsi alors aussi cela ceci celui celle ceux celles encore dès lors qu en y'.split())
IT_STOP = set('il lo la i gli le un uno una di a da in con su per tra fra che cui chi cui dove come quando perché poiché mentre se ma anche ed ebbe è sono sei siamo siete è anch ancora dopo prima durante questo questa questi queste quel quello quella quei quegli quelle non si suo sua suoi sue mio mia miei mie tuo tua tuoi tue nostro nostra nostri nostre vostro vostra vostri vostre loro essere avere fare detto fatto altri altre altro essere stato sopra sotto dentro fuori molto poco più meno tutto tutti tutta tutte ogni ognuno ogniuno entrambi quale quali alcuni alcune'.split())

ALL_STOP = EN_STOP | DE_STOP | FR_STOP | IT_STOP
# Common PDF/document artifacts
ALL_STOP |= {'pdf', 'page', 'fig', 'figure', 'table', 'www', 'http', 'https', 'com', 'org',
             'eu', 'usa', 'uk', 'nan', 'see', 'shall', 'use', 'used', 'using', 'use'}
print(f'Stopwords total: {len(ALL_STOP)}')


## 6. Top Words per Country (Frequency)

In [ ]:
def top_words(text, n=25):
    tokens = [t for t in word_tokenize(text) if t not in ALL_STOP]
    return Counter(tokens).most_common(n)


fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.flatten()

for ax, (country, text) in zip(axes, texts.items()):
    top = top_words(text, n=20)
    words = [w for w, _ in top][::-1]
    counts = [c for _, c in top][::-1]
    ax.barh(words, counts, color=COLORS[country])
    ax.set_title(f'{country} — Top 20 Words', fontsize=11, fontweight='bold')
    ax.tick_params(axis='y', labelsize=8)

plt.tight_layout()
plt.show()


## 7. TF-IDF — What's Distinctive About Each Country?

TF-IDF identifies words that are *unusually frequent* in one document relative to the others.


In [ ]:
docs = [texts[c] for c in texts.keys()]
countries = list(texts.keys())

vec = TfidfVectorizer(
    stop_words=list(ALL_STOP),
    token_pattern=r"[a-zA-Zà-ÿÀ-ÝäöüÄÖÜß']{3,}",
    max_features=2000,
    lowercase=True,
)
tfidf = vec.fit_transform(docs)
vocab = vec.get_feature_names_out()

# Top distinctive words per country
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.flatten()

for ax, (i, country) in zip(axes, enumerate(countries)):
    row = tfidf[i].toarray().flatten()
    top_idx = row.argsort()[-20:][::-1]
    words = [vocab[j] for j in top_idx][::-1]
    scores = [row[j] for j in top_idx][::-1]
    ax.barh(words, scores, color=COLORS[country])
    ax.set_title(f'{country} — Top 20 Distinctive Terms (TF-IDF)',
                 fontsize=11, fontweight='bold')
    ax.tick_params(axis='y', labelsize=8)

plt.tight_layout()
plt.show()


## 8. Document Similarity (Cosine on TF-IDF)

How similar are the strategies to one another?


In [ ]:
sim_matrix = cosine_similarity(tfidf)
sim_df = pd.DataFrame(sim_matrix, index=countries, columns=countries).round(3)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(sim_df.values, cmap='YlOrRd', vmin=0, vmax=1)
ax.set_xticks(range(len(countries)))
ax.set_xticklabels(countries, rotation=30, ha='right')
ax.set_yticks(range(len(countries)))
ax.set_yticklabels(countries)
ax.set_title('Document Similarity (Cosine on TF-IDF)', fontweight='bold')

for i in range(len(countries)):
    for j in range(len(countries)):
        ax.text(j, i, f'{sim_df.values[i,j]:.2f}',
                ha='center', va='center', fontsize=10,
                color='white' if sim_df.values[i,j] > 0.5 else 'black')

plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()
sim_df


## 9. Theme Keyword Tracking

Search how often each strategy mentions specific policy themes.


In [ ]:
# Theme keywords (English roots; will hit multilingual variants by stem-like matching)
THEMES = {
    'Innovation':    ['innovation', 'innov'],
    'Investment':    ['investment', 'investing', 'fund', 'investi'],
    'Ethics':        ['ethics', 'ethical', 'éthique', 'ethik', 'etica'],
    'Regulation':    ['regulation', 'regulator', 'régul', 'regul'],
    'Safety/Risk':   ['safety', 'risk', 'sécurité', 'risque', 'sicherheit', 'risiko', 'rischio'],
    'Talent/Skills': ['talent', 'skill', 'workforce', 'compétences', 'fachkräfte', 'competenze'],
    'Research':      ['research', 'recherche', 'forschung', 'ricerca'],
    'Industry':      ['industry', 'industrial', 'industrie', 'industria'],
    'Privacy/Data':  ['privacy', 'data protection', 'donnée', 'datenschutz', 'privacy'],
    'National Sec.': ['national security', 'defense', 'défense', 'verteidigung', 'difesa'],
}


def count_theme(text, keywords):
    text_lower = text.lower()
    return sum(text_lower.count(kw) for kw in keywords)


theme_counts = pd.DataFrame(
    {country: {theme: count_theme(texts[country], kws) for theme, kws in THEMES.items()}
     for country in countries}
)

# Normalize per 1000 words to compare across documents of different lengths
word_counts = pd.Series({c: len(word_tokenize(texts[c])) for c in countries})
theme_norm = (theme_counts.div(word_counts) * 1000).round(2)

print('Raw counts:')
display(theme_counts)
print('\nPer 1000 words:')
display(theme_norm)


In [ ]:
# Heatmap of normalized theme presence
fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(theme_norm.values, aspect='auto', cmap='YlGnBu')
ax.set_xticks(range(len(countries)))
ax.set_xticklabels(countries, rotation=20, ha='right')
ax.set_yticks(range(len(theme_norm.index)))
ax.set_yticklabels(theme_norm.index)
ax.set_title('Theme Mentions per 1,000 Words', fontweight='bold')

for i in range(len(theme_norm.index)):
    for j in range(len(countries)):
        v = theme_norm.values[i, j]
        ax.text(j, i, f'{v:.1f}', ha='center', va='center',
                fontsize=9, color='white' if v > theme_norm.values.max()*0.5 else 'black')

plt.colorbar(im, ax=ax, label='Mentions per 1000 words')
plt.tight_layout()
plt.show()


## 10. Bigram Analysis — Top Phrases per Country


In [ ]:
bg_vec = CountVectorizer(
    stop_words=list(ALL_STOP),
    token_pattern=r"[a-zA-Zà-ÿÀ-ÝäöüÄÖÜß']{3,}",
    ngram_range=(2, 2),
    max_features=2000,
)
bg = bg_vec.fit_transform(docs)
bg_vocab = bg_vec.get_feature_names_out()

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.flatten()
for ax, (i, country) in zip(axes, enumerate(countries)):
    row = bg[i].toarray().flatten()
    top_idx = row.argsort()[-15:][::-1]
    phrases = [bg_vocab[j] for j in top_idx][::-1]
    counts = [row[j] for j in top_idx][::-1]
    ax.barh(phrases, counts, color=COLORS[country])
    ax.set_title(f'{country} — Top 15 Bigrams', fontsize=11, fontweight='bold')
    ax.tick_params(axis='y', labelsize=8)

plt.tight_layout()
plt.show()


## 11. Topic Modeling (LDA)

Discover latent topics across all 4 documents.


In [ ]:
# CountVectorizer for LDA
ct_vec = CountVectorizer(
    stop_words=list(ALL_STOP),
    token_pattern=r"[a-zA-Zà-ÿÀ-ÝäöüÄÖÜß']{3,}",
    max_features=1500,
    min_df=2,
    max_df=0.95,
)
ct = ct_vec.fit_transform(docs)
ct_vocab = ct_vec.get_feature_names_out()

n_topics = 4
lda = LatentDirichletAllocation(n_components=n_topics, random_state=42, max_iter=20)
doc_topic = lda.fit_transform(ct)

# Print top words per topic
print('Top words per topic:')
for i, topic in enumerate(lda.components_):
    top_words_topic = [ct_vocab[j] for j in topic.argsort()[-12:][::-1]]
    print(f'  Topic {i+1}: {", ".join(top_words_topic)}')

print()
topic_df = pd.DataFrame(doc_topic.round(3), index=countries,
                        columns=[f'Topic {i+1}' for i in range(n_topics)])
print('Topic distribution per document:')
display(topic_df)


In [ ]:
# Stacked bar of topic distribution
fig, ax = plt.subplots(figsize=(10, 5))
bottom = np.zeros(len(countries))
topic_colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
for i in range(n_topics):
    vals = topic_df[f'Topic {i+1}'].values
    ax.bar(countries, vals, bottom=bottom, label=f'Topic {i+1}', color=topic_colors[i])
    bottom += vals

ax.set_ylabel('Topic share')
ax.set_title('Topic Distribution per National AI Strategy', fontweight='bold')
ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5))
plt.tight_layout()
plt.show()


## 12. Save Cleaned Texts and Stats

In [ ]:
out_dir = repo_root / 'data' / 'pdf' / 'AI Policy' / '_extracted'
out_dir.mkdir(parents=True, exist_ok=True)

for country, text in texts.items():
    (out_dir / f'{country}.txt').write_text(text, encoding='utf-8')

stats_df.to_csv(out_dir / 'document_stats.csv')
theme_counts.to_csv(out_dir / 'theme_counts_raw.csv')
theme_norm.to_csv(out_dir / 'theme_per_1000_words.csv')
sim_df.to_csv(out_dir / 'similarity_matrix.csv')

print(f'Saved outputs to: {out_dir}')
for f in sorted(out_dir.iterdir()):
    print(f'  - {f.name}')


## What Else Can You Do?

- **Sentence-level extraction** — pull sentences mentioning specific themes for qualitative review
- **Named entity recognition** (spaCy) — extract organizations, locations, numbers
- **Sentiment analysis** — tone of policy language (more useful with multilingual models)
- **Cross-language alignment** — translate to one language first (DeepL / Google Translate API) for stronger comparison
- **Time-aware analysis** — if you have multiple versions per country (e.g. 2018 vs 2023), track wording changes
- **Citation-style network** — extract references and build a graph of who cites whom
